<a href="https://colab.research.google.com/github/nahmeddn-sys/GENAI/blob/main/Assignment_2_Implement_transfer_learning_using_an_LLM_Mistral_or_Claude_with_a_small_dataset_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Introduction**

Transfer learning is a widely used technique in Generative AI where a pre-trained large language model (LLM) is adapted to perform a specific task using a smaller dataset. Instead of training a model from scratch, transfer learning leverages the knowledge already learned by powerful models such as Mistral-7B. These models are originally trained on massive text datasets and understand language patterns, grammar, and general knowledge. In this project, the pre-trained model is fine-tuned using a small customer support question-answer dataset so that it can respond to queries related to e-commerce services such as order tracking, refunds, and delivery information. By using techniques such as parameter-efficient fine-tuning and running the training process on the free GPU provided by Google Colab, the model can be efficiently adapted for a specific domain without requiring expensive computing resources.

**Explanation**

In this implementation, transfer learning is performed by taking the pre-trained Mistral model and adapting it to a new task using a structured dataset of instructions and responses. The dataset contains realistic customer support queries along with their corresponding answers. During training, the model learns the relationship between the instruction (question) and the expected response. Instead of modifying all parameters of the large model, lightweight fine-tuning methods are used so that only a small subset of parameters are updated. This approach reduces memory usage and makes it possible to train the model on limited hardware such as Google Colab’s free GPU environment. After training, the model can generate relevant responses to new customer support questions by applying the patterns it learned from the dataset, demonstrating how transfer learning allows large language models to be efficiently customized for real-world applications

**Install Libraries **

In [ ]:
!pip install unsloth
!pip install transformers datasets accelerate peft trl bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 132.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 128.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**Load Mistral Model**

-Mistral-7B is loaded in 4-bit quantization

-This reduces GPU memory usage

-Makes training possible on Colab T4 GPU

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/mistral-7b-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

**Apply Transfer Learning (LoRA)**

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.3 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


**Customer Support Dataset (100 Samples)**

In [ ]:
from datasets import Dataset

data= [
{"instruction":"Where is my order?","response":"You can track your order using the tracking link sent to your email after shipment."},
{"instruction":"How long does delivery take?","response":"Standard delivery usually takes 3 to 5 business days depending on your location."},
{"instruction":"Can I cancel my order?","response":"Yes, you can cancel your order before it is shipped from the warehouse."},
{"instruction":"How do I return a product?","response":"You can initiate a return request from your account under the orders section."},
{"instruction":"What is your return policy?","response":"We offer a 7 day return policy for most products if they are unused and in original packaging."},
{"instruction":"I received a damaged product. What should I do?","response":"Please upload photos of the damaged product and request a replacement through the returns portal."},
{"instruction":"Do you offer cash on delivery?","response":"Yes, cash on delivery is available for selected locations."},
{"instruction":"How can I change my delivery address?","response":"You can update your address before the order is shipped by visiting your order details page."},
{"instruction":"My payment failed but money was deducted.","response":"If the payment was deducted, it will automatically be refunded within 5 to 7 business days."},
{"instruction":"How do I apply a discount coupon?","response":"You can apply a coupon code during checkout in the coupon section."},

{"instruction":"Do you ship internationally?","response":"Currently we ship only within the country."},
{"instruction":"What payment methods do you accept?","response":"We accept credit cards, debit cards, UPI and net banking."},
{"instruction":"Can I track my shipment?","response":"Yes, tracking details will be sent to your email and SMS once the order is shipped."},
{"instruction":"My order is delayed.","response":"We apologize for the delay. Please check the tracking link for the latest delivery update."},
{"instruction":"How do I contact customer support?","response":"You can contact our support team through chat, email or phone."},
{"instruction":"Can I exchange my product?","response":"Yes, exchanges are available for size or color within 7 days of delivery."},
{"instruction":"How do I reset my account password?","response":"Click on the forgot password option on the login page and follow the instructions."},
{"instruction":"I entered the wrong address.","response":"Please contact support immediately so we can update the address before shipping."},
{"instruction":"Why was my order cancelled?","response":"Orders may be cancelled due to payment issues or stock unavailability."},
{"instruction":"Can I place bulk orders?","response":"Yes, bulk orders can be placed by contacting our sales team."},

{"instruction":"Do you provide warranty on products?","response":"Warranty depends on the manufacturer and product category."},
{"instruction":"How will I know if my order is confirmed?","response":"You will receive an order confirmation email after successful payment."},
{"instruction":"Can I schedule delivery?","response":"Currently scheduled delivery is not available."},
{"instruction":"Do you provide gift wrapping?","response":"Yes, gift wrapping options are available during checkout."},
{"instruction":"What happens if I miss my delivery?","response":"The courier will attempt delivery again or contact you for rescheduling."},
{"instruction":"Can I update my phone number?","response":"Yes, you can update your phone number in your account settings."},
{"instruction":"Do you offer refunds?","response":"Yes, refunds are processed after the returned item is inspected."},
{"instruction":"How long does refund take?","response":"Refunds are usually processed within 5 to 7 business days."},
{"instruction":"Where can I see my order history?","response":"Your order history is available in the orders section of your account."},
{"instruction":"Do you have a mobile app?","response":"Yes, our mobile app is available on Android and iOS."},

{"instruction":"Can I modify my order after placing it?","response":"Order modifications are allowed before shipment."},
{"instruction":"What if the product is out of stock?","response":"You can sign up for notifications when the product becomes available again."},
{"instruction":"How do I delete my account?","response":"Please contact customer support to request account deletion."},
{"instruction":"Do you provide installation services?","response":"Installation services are available for selected products."},
{"instruction":"Can I get an invoice for my purchase?","response":"Yes, invoices can be downloaded from the orders section."},
{"instruction":"What is express delivery?","response":"Express delivery ensures faster shipping within 1 to 2 business days."},
{"instruction":"Why is my order still processing?","response":"Processing means the order is being prepared for shipment."},
{"instruction":"Do you offer student discounts?","response":"Occasionally we offer promotional discounts for students."},
{"instruction":"Can I pre order products?","response":"Yes, pre orders are available for selected upcoming products."},
{"instruction":"What should I do if I received the wrong item?","response":"Please initiate a return request and we will arrange a replacement."},

{"instruction":"How do I subscribe to your newsletter?","response":"You can subscribe by entering your email on our website homepage."},
{"instruction":"Are there any shipping charges?","response":"Shipping charges depend on the order value and delivery location."},
{"instruction":"Is there free shipping?","response":"Yes, free shipping is available on orders above a certain amount."},
{"instruction":"Do you sell gift cards?","response":"Yes, digital gift cards are available on our website."},
{"instruction":"Can gift cards be refunded?","response":"Gift cards are non refundable once purchased."},
{"instruction":"How do I check my refund status?","response":"You can check refund status in the order details section."},
{"instruction":"Do you provide order tracking updates?","response":"Yes, tracking updates are sent via SMS and email."},
{"instruction":"Can I change product size after ordering?","response":"Size changes can be requested before shipment."},
{"instruction":"Do you offer loyalty rewards?","response":"Yes, customers earn reward points on every purchase."},
{"instruction":"How do I redeem reward points?","response":"Reward points can be redeemed during checkout."},

{"instruction":"What should I do if my order is lost?","response":"Please contact support so we can investigate with the courier."},
{"instruction":"How can I update my email address?","response":"Email addresses can be updated in account settings."},
{"instruction":"Can I reorder a previous purchase?","response":"Yes, you can reorder items from your order history."},
{"instruction":"Do you provide product recommendations?","response":"Yes, recommendations are available based on browsing history."},
{"instruction":"Is my payment information secure?","response":"Yes, we use secure encryption for all transactions."},
{"instruction":"Can I save items for later?","response":"Yes, you can add items to your wishlist."},
{"instruction":"Do you provide customer reviews?","response":"Yes, customers can read and write reviews on product pages."},
{"instruction":"How do I report a problem with the website?","response":"Please contact support with details of the issue."},
{"instruction":"Can I buy products in installments?","response":"Yes, installment options are available through selected payment partners."},
{"instruction":"What browsers are supported?","response":"Our website works on most modern browsers including Chrome and Firefox."},

{"instruction":"Can I update my delivery instructions?","response":"Delivery instructions can be added in the order details page."},
{"instruction":"How do I unsubscribe from emails?","response":"Click the unsubscribe link at the bottom of our emails."},
{"instruction":"Do you offer seasonal sales?","response":"Yes, we offer special discounts during festive seasons."},
{"instruction":"How do I report fraudulent activity?","response":"Please contact support immediately if you notice suspicious activity."},
{"instruction":"Can I request faster delivery?","response":"You may select express shipping if available in your area."},
{"instruction":"Are product prices negotiable?","response":"Prices listed on the website are fixed."},
{"instruction":"Can I request product customization?","response":"Customization is available for selected products only."},
{"instruction":"Do you provide technical support?","response":"Yes, technical support is available for electronic products."},
{"instruction":"How do I check product availability?","response":"Product availability is shown on the product page."},
{"instruction":"Can I request product recommendations?","response":"Our support team can suggest products based on your needs."},

{"instruction":"Why is my coupon not working?","response":"Please check if the coupon has expired or minimum purchase requirements are met."},
{"instruction":"Can I combine multiple coupons?","response":"Only one coupon can be applied per order."},
{"instruction":"What if I receive an incomplete order?","response":"Please report the issue and we will investigate immediately."},
{"instruction":"Do you offer business accounts?","response":"Yes, business accounts are available for bulk buyers."},
{"instruction":"How do I download the invoice?","response":"Invoices can be downloaded from your order details page."},
{"instruction":"Can I add items to an existing order?","response":"Items cannot be added after an order is placed."},
{"instruction":"What if the courier cannot find my address?","response":"The courier will contact you using your phone number."},
{"instruction":"Do you offer weekend delivery?","response":"Weekend delivery depends on courier availability."},
{"instruction":"How do I update my profile information?","response":"Profile information can be updated in account settings."},
{"instruction":"What happens if my payment is declined?","response":"You can try another payment method or contact your bank."}
]
data = data * 5
dataset = Dataset.from_list(data)

**Format Dataset**

In [ ]:
EOS_TOKEN = tokenizer.eos_token

def format_prompt(example):
    text = f"""### Instruction:
{example['instruction']}

### Response:
{example['response']}{EOS_TOKEN}"""

    return {"text": text}

dataset = dataset.map(format_prompt)
eos_token_id = tokenizer.eos_token_id

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

**Training Configuration**

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir="outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/400 [00:00<?, ? examples/s]

**Train Model**

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 7,255,363,584 (0.19% trained)


Step,Training Loss
1,0.667017
2,0.599490
3,0.575573
4,0.412302
5,0.316587
6,0.322894
7,0.397637
8,0.356800
9,0.334917
10,0.357709


TrainOutput(global_step=60, training_loss=0.3067867693801721, metrics={'train_runtime': 113.63, 'train_samples_per_second': 4.224, 'train_steps_per_second': 0.528, 'total_flos': 708895509774336.0, 'train_loss': 0.3067867693801721, 'epoch': 1.2})

**Test the Model**

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

question = """### Instruction:
How can I track my delivery?

### Response:
"""

inputs = tokenizer(question, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    temperature=0.1,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

### Instruction:
How can I track my delivery?

### Response:
You will receive tracking details via SMS and email once the order is shipped.
